In [7]:
# Imports

!pip install kagglehub
!pip install gensim
import kagglehub
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer

In [8]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Download required NLTK data
nltk.download('wordnet', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)  # for tokenization
nltk.download('averaged_perceptron_tagger', quiet=True)  # for POS tagging

# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()

# Create custom stop words set (preserving negations)
stop_words = set(stopwords.words('english'))
negation_words = {'not', 'no', 'nor', 'never', 'neither', 'nothing', 'nowhere', 'nobody'}
stop_words = stop_words - negation_words

# Example preprocessing function
def preprocess_text(text):
    tokens = word_tokenize(text.lower())
    filtered_tokens = [lemmatizer.lemmatize(token) for token in tokens 
                      if token.isalpha() and token not in stop_words]
    return filtered_tokens

## Data Processing

In [9]:
# Loading Dataset (FULL)

path = kagglehub.dataset_download("jp797498e/twitter-entity-sentiment-analysis")
df = pd.read_csv(path + "/twitter_training.csv", header=None)

In [10]:
# Data Preprocessing
df.columns = ['id', 'entity', 'sentiment', 'text']
data = df[['text', 'sentiment']].dropna().copy()

# removeing noisy labels using boolean mask
data = data[data['sentiment'] != 'irrelevant']

# Cleaning
def clean_text(text):
    text = re.sub(r'<.*?>', '', str(text))
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    # Splitting + Tokenization
    words = text.split()

    words = [
        lemmatizer.lemmatize(w, pos='v')
        for w in words
        if w not in stop_words
    ]

    return words

tokenized = [clean_text(t) for t in data['text']]
texts_joined = [" ".join(t) for t in tokenized]

In [11]:
# TF-IDF 
tfidf = TfidfVectorizer(
    max_features=6000,   
    ngram_range=(1,2)
)
X_tfidf = tfidf.fit_transform(texts_joined)

In [12]:
# Word2Vec
w2v = Word2Vec(
    sentences=tokenized,
    vector_size=120,
    window=5,
    min_count=2,
    workers=2
)

def get_vector(words):
    vectors = [w2v.wv[w] for w in words if w in w2v.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(120)

X_w2v = np.array([get_vector(t) for t in tokenized])

In [13]:
# Combining features

X = np.hstack([X_w2v, X_tfidf.toarray()])
y = data['sentiment'].astype('category').cat.codes

In [14]:
# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## Model Training

#### Random Forest

In [15]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=30,
    max_features='sqrt',
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))

Random Forest Accuracy: 0.7431081081081081


### ANN

In [16]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

ann = MLPClassifier(
    hidden_layer_sizes=(256,128,64),
    max_iter=400,
    early_stopping=True,
    learning_rate='adaptive',
    random_state=42
)

ann.fit(X_train_s, y_train)
ann_pred = ann.predict(X_test_s)

print("ANN Accuracy:", accuracy_score(y_test, ann_pred))

ANN Accuracy: 0.8461486486486487


## Random Forest Accuracy: 0.7431081081081081
## ANN Accuracy: 0.8461486486486487